In [0]:
from pyspark.sql.functions import current_timestamp, from_unixtime, from_json

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType

schema = StructType([
    StructField("name", StringType()),
    StructField("timezone", IntegerType()),
    StructField("dt", IntegerType()),
    StructField("main", StructType([
        StructField("temp", DoubleType()),
        StructField("feels_like", DoubleType()),
        StructField("temp_min", DoubleType()),
        StructField("temp_max", DoubleType()),
        StructField("humidity", IntegerType()),
        StructField("pressure", IntegerType()),
    ])),
    StructField("weather", ArrayType(StructType([
        StructField("id", IntegerType()),
        StructField("main", StringType()),
        StructField("description", StringType()),
        StructField("icon", StringType()),
    ]))),
    StructField("wind", StructType([
        StructField("speed", DoubleType()),
        StructField("deg", IntegerType()),
    ])),
    StructField("coord", StructType([
        StructField("lat", DoubleType()),
        StructField("lon", DoubleType()),
    ])),
    StructField("sys", StructType([
        StructField("country", StringType()),
        StructField("sunrise", IntegerType()),
        StructField("sunset", IntegerType()),
    ])),
])


In [0]:
df_bronze = spark.read.table("workspace.default.bronze_weather").dropDuplicates(["raw_json"])

df_silver = df_bronze.withColumn("data", from_json("raw_json", schema)) \
                     .select("ingestion_time", "data.*") \
                     .withColumn("datetime", from_unixtime("datetime").cast("timestamp"))

df_silver.write.format("delta").mode("append").saveAsTable("workspace.default.silver_weather")